# Assignment 14

Below is our rule-based implementation of Ok or Ugly classifier within our frontend.
It pushes blocking or warnings based on severity of faults detected, if blocking is pushed and not resolved it can not in any cirumstance be saved and pushed to the backend pipeline but if resolved and more the $65\%$ of frames are OK it is allowed to be forwarded to backend pipeline, if warning is pushed it can be saved and pushed to the backend pipeline in current implementation format.

| Type | What | How |
| :--- | :--- | :--- |
| **Blocking** | Too dark | Mean luminance less than 28 on canvas pixels |
| **Blocking** | Overexposed | Mean luminance greater than 238 on canvas pixels |
| **Warning** | Upside down | Nose y-coordinate lower than ankles |
| **Warning** | Hands below hips | Wrist y greater than hip y by a margin |
| **Warning** | Person out of frame | 2+ key landmarks near the edge (less than 4%) |
| **Warning** | Person too far away | Body bounding box less than 30% of frame height |
| **Warning** | Facing camera head-on | Knees invisible but shoulders clearly visible |

```javascript
/**
 * Assess per-frame video quality for squat analysis.
 * Returns { blocking, warnings } where:
 *   blocking — issues that prevent reliable analysis (pipeline should be gated)
 *   warnings — advisory hints shown live but never block the pipeline
 */
function assessVideoQuality(detection, captureCanvas) {
    if (!detection?.landmarks?.length) {
        return { blocking: ["No person detected in frame"], warnings: [] };
    }

    const lms = detection.landmarks[0];
    const blocking = [];
    const warnings = [];
    const CORE = [11, 12, 23, 24, 25, 26, 27, 28];
    const vis = (i) => lms[i]?.visibility ?? 0;

    // BLOCKING: frame brightness
    if (captureCanvas?.width > 0 && captureCanvas?.height > 0) {
        try {
            const ctx = captureCanvas.getContext("2d");
            const { width: w, height: h } = captureCanvas;
            const { data } = ctx.getImageData(
                Math.floor(w * 0.25),
                Math.floor(h * 0.15),
                Math.floor(w * 0.5),
                Math.floor(h * 0.7),
            );
            let lum = 0;
            for (let i = 0; i < data.length; i += 4) {
                lum +=
                    0.299 * data[i] + 0.587 * data[i + 1] + 0.114 * data[i + 2];
            }
            const avg = lum / (data.length / 4);
            // TODO: Improve values for better preformance
            if (avg < 28)
                blocking.push("Frame too dark: improve room lighting");
            else if (avg > 238)
                blocking.push(
                    "Frame overexposed: reduce direct lighting behind the subject",
                );
        } catch {
            /* cross-origin / security errors ignored */
        }
    }

    // WARNINGS: advisory hints — never stop the pipeline

    // Upside down
    const nose = lms[0],
        la = lms[27],
        ra = lms[28];
    if (nose && la && ra && nose.y > (la.y + ra.y) / 2 + 0.1) {
        warnings.push("Video appears upside down: rotate the camera 180°");
    }

    // Hands below hips (TODO: Not working as intended)
    const lw = lms[15],
        rw = lms[16],
        lh = lms[23],
        rh = lms[24];
    if (lw && rw && lh && rh && vis(15) > 0.5 && vis(16) > 0.5) {
        if ((lw.y + rw.y) / 2 > (lh.y + rh.y) / 2 + 0.12) {
            warnings.push(
                "Hands are below the hips: unusual posture for squat analysis",
            );
        }
    }

    // Key joints clipped at frame edges
    const coreLms = CORE.map((i) => lms[i]).filter(Boolean);
    if (
        coreLms.filter(
            (lm) => lm.x < 0.04 || lm.x > 0.96 || lm.y < 0.04 || lm.y > 0.96,
        ).length >= 2
    ) {
        warnings.push(
            "Person is partially out of frame: move camera back or reposition",
        );
    }

    // Person too small / too far (TODO: Too limiting)
    const ys = coreLms.map((lm) => lm.y);
    const xs = coreLms.map((lm) => lm.x);
    if (
        ys.length >= 4 &&
        Math.max(...ys) - Math.min(...ys) < 0.3 &&
        Math.max(...xs) - Math.min(...xs) < 0.2
    ) {
        warnings.push("Person is too far from the camera: move closer");
    }

    // Knees hidden while shoulders clearly visible (facing camera head-on)
    if (vis(25) < 0.35 && vis(26) < 0.35 && vis(11) > 0.6 && vis(12) > 0.6) {
        if (Math.abs((lms[12]?.x ?? 0) - (lms[11]?.x ?? 0)) > 0.15) {
            warnings.push(
                "Knees not visible: try a side-on camera angle for better depth measurement",
            );
        }
    }

    return { blocking, warnings };
}
```